In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(df.shape)
df.head()

In [ ]:
print(df.isnull().sum())
print(df['SeriousDlqin2yrs'].value_counts())

In [ ]:
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())
print(df.isnull().sum())

In [ ]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:,1]
print('AUC:', roc_auc_score(y_test, y_pred))

In [ ]:
# Score scaling: map probability to credit score (300-850)
def prob_to_score(prob, pdo=20, base_score=600, base_odds=50):
    factor = pdo / np.log(2)
    offset = base_score - factor * np.log(base_odds)
    score = offset - factor * np.log(prob / (1 - prob + 1e-10))
    return np.clip(score, 300, 850)

scores = prob_to_score(y_pred)
plt.figure(figsize=(10, 5))
plt.hist(scores, bins=50, edgecolor='black')
plt.title('Credit Score Distribution')
plt.xlabel('Score')
plt.ylabel('Count')
plt.show()

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='Coefficient', y='Feature')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()